# Semana 8 · Sesión 2: `KanesMethod` y simulación

**Módulo 2**

## Objetivos de la sesión

1. Plantear un sistema con velocidades generalizadas y ecuaciones cinemáticas,
   y obtener sus ecuaciones de movimiento con `KanesMethod`.
2. Comprobar que Kane y Lagrange dan las mismas ecuaciones de movimiento.
3. Convertir las ecuaciones simbólicas en una simulación con `lambdify` y
   `solve_ivp`, y validarla con la conservación de la energía.

## Retomamos

La sesión pasada `LagrangesMethod` generó las ecuaciones de movimiento del
péndulo doble a partir de su Lagrangiano, y `rhs()` las dejó en primer orden,
listas para un integrador. Hoy hacemos dos cosas con ellas: obtenerlas **por
otra ruta**, la formulación de Kane, y **resolverlas** numéricamente.

La celda de abajo reconstruye, ya resuelto, el péndulo doble de la sesión 1
—cinemática, partículas, Lagrangiano y ecuaciones—, para que este notebook
corra por sí solo. Sus marcos y puntos se llaman `B1`, `B2`, `Q1`, `Q2` para
dejarte libres `A1`, `A2`, `P1`, `P2`. Se agregan NumPy, Matplotlib y
`solve_ivp`, el integrador de SciPy.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
import sympy.physics.mechanics as me
from IPython.display import Math
from scipy.integrate import solve_ivp

me.init_vprinting()

t = sp.Symbol("t")               # el tiempo de mechanics: sin suposiciones
N = me.ReferenceFrame("N")       # el marco del laboratorio
O = me.Point("O")
O.set_vel(N, 0)                  # el pivote, fijo en el laboratorio

# El péndulo doble de la sesión 1, con LagrangesMethod.
theta1, theta2 = me.dynamicsymbols("theta1 theta2")
m1, m2, L1, L2, g = sp.symbols("m1 m2 L1 L2 g", positive=True)

B1 = me.ReferenceFrame("B1")
B1.orient_axis(N, N.z, theta1)
B2 = me.ReferenceFrame("B2")
B2.orient_axis(N, N.z, theta2)
Q1 = O.locatenew("Q1", -L1 * B1.y)
Q2 = Q1.locatenew("Q2", -L2 * B2.y)

masa_1 = me.Particle("masa_1", Q1, m1)
masa_2 = me.Particle("masa_2", Q2, m2)
masa_1.potential_energy = m1 * g * Q1.pos_from(O).dot(N.y)
masa_2.potential_energy = m2 * g * Q2.pos_from(O).dot(N.y)

metodo_lagrange = me.LagrangesMethod(me.Lagrangian(N, masa_1, masa_2), [theta1, theta2])
metodo_lagrange.form_lagranges_equations()
rhs_lagrange = metodo_lagrange.rhs()

## ¿Por qué otra formulación?

Lagrange es elegante, pero tiene dos incomodidades que ya asomaron la sesión
pasada: todo pasa por la **energía**, así que las fuerzas no conservativas se
llevan por fuera; y derivar el Lagrangiano completo de un sistema con muchos
cuerpos produce expresiones enormes que luego hay que simplificar.

El método de **Kane** trabaja directamente con **fuerzas y aceleraciones**,
proyectadas sobre las direcciones en que el sistema se puede mover. No
necesita energía potencial: la gravedad, la fricción o un motor entran igual,
como fuerzas. Es el método preferido de la propia documentación de
`sympy.physics.mechanics`, y el que se usa en robótica y biomecánica.

| | Lagrange | Kane |
|---|---|---|
| Parte de | $\mathcal{L} = T - V$ | Fuerzas y aceleraciones |
| Fricción | Por fuera, en `forcelist` | Una fuerza más |
| Incógnitas | $q$, $\dot q$ | $q$ y **velocidades generalizadas** $u$ |
| Resultado | Las mismas ecuaciones de movimiento | Las mismas ecuaciones de movimiento |

## Velocidades generalizadas

Kane pide nombrar aparte las velocidades: una **velocidad generalizada** $u$
por cada coordenada. En los casos de este curso es simplemente $u = \dot q$,
pero hay que decirlo explícitamente, con una **ecuación cinemática**
$\dot q - u = 0$. (En sistemas más complicados conviene escoger otra $u$: esa
libertad es parte de la potencia del método.)

Con el péndulo simple. La novedad está en `set_ang_vel`: le decimos al marco
que su velocidad angular es $u\,\hat{n}_z$, en lugar de dejar que la calcule
como $\dot\theta\,\hat{n}_z$. Así todas las velocidades quedan escritas con $u$.

In [ ]:
theta, u = me.dynamicsymbols("theta u")
m, L = sp.symbols("m L", positive=True)       # masa y longitud

A = me.ReferenceFrame("A")
A.orient_axis(N, N.z, theta)
A.set_ang_vel(N, u * N.z)                     # la velocidad angular, en términos de u

P = me.Point("P")
P.set_pos(O, -L * A.y)
display(Math(r"{}^N\mathbf{v}^P = " + me.vlatex(P.v2pt_theory(O, N, A))))

## `KanesMethod`

| Paso | Escribe |
|---|---|
| Plantear | `me.KanesMethod(N, q_ind=[q], u_ind=[u], kd_eqs=[q.diff(t) - u])` |
| Generar las ecuaciones | `fr, frstar = metodo.kanes_equations(cuerpos, cargas)` |
| Primer orden | `metodo.rhs()` |

Las **cargas** son una lista de pares: `(punto, fuerza)` para una fuerza
aplicada en un punto, o `(marco, torque)` para un torque sobre un cuerpo.
`kanes_equations` devuelve dos columnas: $F_r$, las **fuerzas generalizadas**
que vienen de las cargas, y $F_r^*$, las **fuerzas de inercia**, que vienen de
masas por aceleraciones. La ecuación de movimiento es $F_r + F_r^* = 0$.

In [ ]:
bola = me.Particle("bola", P, m)
cargas = [(P, -m * g * N.y)]                  # el peso, aplicado en la masa

metodo_kane = me.KanesMethod(N, q_ind=[theta], u_ind=[u], kd_eqs=[theta.diff(t) - u])
fr, frstar = metodo_kane.kanes_equations([bola], cargas)

display(sp.Eq(sp.Symbol("F_r"), fr[0]))
display(sp.Eq(sp.Symbol("F_r^*"), frstar[0]))
display(sp.Eq(fr[0] + frstar[0], 0))

Es la ecuación de la sesión 1 con el signo cambiado y $\dot u$ en lugar de
$\ddot\theta$: $F_r$ es el torque del peso y $F_r^*$ es menos el momento de
inercia por la aceleración angular. Nota que **no** escribimos ninguna energía
potencial: el peso entró como fuerza.

`rhs()` da el sistema de primer orden, con estado $(\theta, u)$. Sale
idéntico al de Lagrange.

In [ ]:
display(sp.Eq(sp.Symbol(r"\dot{\mathbf{x}}"), metodo_kane.rhs(), evaluate=False))

## Fricción: una fuerza más en la lista

Con Kane no hace falta ningún mecanismo aparte para la fricción viscosa: es
otra fuerza aplicada en el punto, y se suma al peso.

In [ ]:
b = sp.Symbol("b", positive=True)   # coeficiente de fricción viscosa

cargas_con_friccion = [(P, -m * g * N.y - b * P.vel(N))]

metodo_friccion = me.KanesMethod(N, q_ind=[theta], u_ind=[u], kd_eqs=[theta.diff(t) - u])
metodo_friccion.kanes_equations([bola], cargas_con_friccion)

display(sp.Eq(sp.Derivative(u, t), sp.expand(metodo_friccion.rhs()[1])))

## El error más común: olvidar las ecuaciones cinemáticas

Sin `kd_eqs`, `KanesMethod` no sabe cómo se relacionan $q$ y $u$, y no puede
formar las ecuaciones. El mensaje no es muy claro —habla de *kinematic
differential equations*—, así que conviene reconocerlo.

In [ ]:
sin_cinematica = me.KanesMethod(N, q_ind=[theta], u_ind=[u])

try:
    sin_cinematica.kanes_equations([bola], cargas)
except AttributeError as error:
    print("AttributeError -", error)

## TODO en clase 1

El péndulo doble con Kane, y la prueba de que da lo mismo que Lagrange.

1. Declara `u1` y `u2` como `dynamicsymbols`.
2. Crea los marcos `A1` y `A2` orientados como en la sesión 1, y fija su
   velocidad angular con `set_ang_vel` en términos de `u1` y `u2`.
3. Ubica `P1` y `P2` y calcula sus velocidades con `v2pt_theory`, encadenadas
   como la semana pasada.
4. Crea las partículas `particula_1` y `particula_2` (masas `m1` y `m2`) y la
   lista `cargas_doble` con el peso de cada una.
5. Plantea `metodo_kane_doble` con las dos coordenadas, las dos velocidades y
   las dos ecuaciones cinemáticas, genera las ecuaciones y calcula
   `rhs_kane = metodo_kane_doble.rhs()`.
6. Compara con `rhs_lagrange`, de la primera celda. Lagrange escribe
   $\dot\theta_i$ donde Kane escribe $u_i$: sustituye antes de restar,
   ```python
   sp.simplify(rhs_kane - rhs_lagrange.subs({theta1.diff(t): u1, theta2.diff(t): u2}))
   ```
   Tiene que salir la matriz cero. Dos caminos, las mismas ecuaciones.

In [ ]:
# TODO en clase: el péndulo doble con Kane
u1 = ...
u2 = ...

A1 = ...
A2 = ...

P1 = ...
P2 = ...

particula_1 = ...
particula_2 = ...
cargas_doble = ...

metodo_kane_doble = ...
rhs_kane = ...

## De las ecuaciones a los números

Ya tenemos $\dot{\mathbf{x}} = F(\mathbf{x})$ en forma simbólica. Para
integrarla, el camino es el de la semana 6 más un paso:

| Paso | Herramienta |
|---|---|
| Fijar los parámetros | `rhs.subs(parametros)` |
| Convertir a función numérica | `sp.lambdify(estado, rhs, "numpy")` |
| Integrar en el tiempo | `solve_ivp(campo, (t0, tf), x0, ...)` |
| Graficar | Matplotlib |

`solve_ivp` pide una función `campo(tiempo, x)` que reciba el tiempo y el
vector de estado y devuelva $\dot{\mathbf{x}}$ como un arreglo plano. Usamos
`rhs_lagrange`: su estado es $(\theta_1, \theta_2, \dot\theta_1, \dot\theta_2)$,
y `lambdify` acepta las derivadas $\dot\theta_i$ como argumentos, igual que
los símbolos.

In [ ]:
parametros = {m1: 1, m2: 1, L1: 1, L2: 1, g: 9.81}   # kg, m, m/s^2
estado = [theta1, theta2, theta1.diff(t), theta2.diff(t)]

rhs_numerica = sp.lambdify(estado, rhs_lagrange.subs(parametros), "numpy")


def campo(tiempo, x):
    """Derivada del estado del péndulo doble, en la forma que pide solve_ivp."""
    return np.asarray(rhs_numerica(*x), dtype=float).ravel()


display(sp.Eq(sp.Symbol(r"\dot{\mathbf{x}}(\pi/2, \pi/2, 0, 0)"),
              sp.Matrix(campo(0, [np.pi/2, np.pi/2, 0, 0])), evaluate=False))

La prueba de arriba: las dos varillas horizontales y en reposo. Las velocidades
son cero, la primera varilla acelera hacia abajo con $\ddot\theta_1 = -g/L_1$
—caída libre— y la segunda, en ese instante, no gira: cae junto con la
primera. Tiene sentido.

Ahora sí, a integrar. `solve_ivp` elige su paso solo; `rtol` y `atol` le dicen
qué tan preciso ser. Con `t_eval` le pedimos la solución en los tiempos que
queremos graficar.

In [ ]:
x0 = [np.pi/2, np.pi/2, 0, 0]              # las dos varillas horizontales, en reposo
tiempos = np.linspace(0, 20, 4000)

solucion = solve_ivp(campo, (0, 20), x0, t_eval=tiempos, rtol=1e-10, atol=1e-10)
print(solucion.message)

In [ ]:
fig, ax = plt.subplots()
ax.plot(solucion.t, solucion.y[0], label=r"$\theta_1$")
ax.plot(solucion.t, solucion.y[1], label=r"$\theta_2$")
ax.set_xlabel("t (s)")
ax.set_ylabel("ángulo (rad)")
ax.set_title("Péndulo doble: ecuaciones de movimiento integradas")
ax.legend()
plt.show()

Nada que ver con las dos oscilaciones suaves que le impusimos la semana pasada.
Veamos la punta: su posición la tenemos simbólica desde la primera celda, así
que es otro `lambdify`.

In [ ]:
punta = Q2.pos_from(O).to_matrix(N).subs(parametros)
x_punta = sp.lambdify([theta1, theta2], punta[0], "numpy")
y_punta = sp.lambdify([theta1, theta2], punta[1], "numpy")

fig, ax = plt.subplots()
ax.plot(x_punta(solucion.y[0], solucion.y[1]), y_punta(solucion.y[0], solucion.y[1]),
        linewidth=0.6)
ax.plot([0], [0], "ko")                        # el pivote
ax.set_aspect("equal")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("Punta del péndulo doble, ahora como la física manda")
plt.show()

## Caos

El péndulo doble es el ejemplo clásico de un sistema **caótico**: dos
condiciones iniciales casi iguales terminan en movimientos completamente
distintos. Probémoslo moviendo el ángulo inicial de la primera varilla una
millonésima de radián.

In [ ]:
x0_vecino = [np.pi/2 + 1e-6, np.pi/2, 0, 0]
vecino = solve_ivp(campo, (0, 20), x0_vecino, t_eval=tiempos, rtol=1e-10, atol=1e-10)

fig, ax = plt.subplots()
ax.semilogy(tiempos, np.abs(vecino.y[0] - solucion.y[0]))
ax.set_xlabel("t (s)")
ax.set_ylabel(r"$|\Delta\theta_1|$ (rad)")
ax.set_title("Dos péndulos que empezaron a una millonésima de radián")
plt.show()

Durante unos segundos la diferencia se queda del tamaño de la perturbación;
después crece exponencialmente —en promedio, una recta en escala
logarítmica— hasta ser del tamaño del ángulo mismo. Las ecuaciones son exactas y deterministas;
lo que no es posible es **predecir** a largo plazo, porque cualquier error en
la condición inicial se amplifica. SymPy nos dio las ecuaciones sin un solo
error de álgebra; el caos está en la física, no en el cálculo.

## TODO en clase 2

¿Cómo sabemos que la simulación es correcta, si el movimiento es caótico y no
hay solución exacta con qué comparar? Sin fricción, la **energía total** se
conserva: es una prueba que la simulación tiene que pasar.

1. Arma `energia`, la energía total simbólica
   $E = T + V$ del péndulo doble, con `me.kinetic_energy(N, masa_1, masa_2)`
   y `me.potential_energy(masa_1, masa_2)` (las partículas de la primera
   celda), y sustituye `parametros`.
2. Conviértela con `lambdify` en `energia_numerica`, con los mismos
   argumentos `estado` que `rhs_numerica`, y evalúala sobre `solucion.y`
   (pasa `*solucion.y`).
3. Grafica `variacion`, $E(t) - E(0)$. Con `rtol=1e-10` su valor absoluto
   máximo debe quedar por debajo de $10^{-6}$ J.
4. Repite la integración con las tolerancias por defecto de `solve_ivp` (sin
   `rtol` ni `atol`, y sin `t_eval`). ¿De qué orden es ahora la variación de
   la energía? ¿Le creerías a esa simulación?

In [ ]:
# TODO en clase: la energía como prueba de la simulación
energia = ...
energia_numerica = ...

variacion = ...

## Resumen

Hoy obtuvimos las ecuaciones de movimiento por una segunda ruta. `KanesMethod`
trabaja con **velocidades generalizadas** $u$, unidas a las coordenadas por las
ecuaciones cinemáticas `kd_eqs`, y con **cargas**: fuerzas en puntos y torques
en marcos. No pide energía potencial, y la fricción es una fuerza más.
Resuelto el péndulo doble por los dos caminos, la diferencia de los `rhs` es
cero.

Después cruzamos de lo simbólico a lo numérico: `subs` fija los parámetros,
`lambdify` convierte `rhs()` en una función, y `solve_ivp` la integra. El
péndulo doble resultó caótico, y la conservación de la energía nos dio una
forma de confiar en la simulación —o de no confiar, si las tolerancias son
laxas—.

**Tarea de la semana:** [`tarea/tarea-08.ipynb`](../tarea/tarea-08.ipynb) —
la cuenta en el aro que gira, ahora con dinámica: sus ecuaciones de movimiento
por Lagrange y por Kane, y sus posiciones de equilibrio.

**Próxima semana — Semana 9:** otros submódulos de física. Unidades y
dimensiones con `sympy.physics.units`, kets y operadores con
`sympy.physics.quantum`, y una primera mirada a la geometría diferencial con
`sympy.diffgeom`.